In [2]:
import sys, os
import datetime
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

In [5]:
import data_aosi
import brack2bing
import crefl2slc

In [4]:
# parameters
t  = torch.linspace(0, 5, 20,      device='cuda')
T  = torch.linspace(-20,10,20,     device='cuda')
S  = torch.linspace(0,35,20,       device='cuda')
rh = torch.linspace(0.6, 0.967,20, device='cuda')
rf = torch.linspace(0,  1,     20, device='cuda')
#mag = torch.linspace(0, 1,     20, device='cuda')
#phs = torch.linspace(0, 1,     20, device='cuda')
mesh = torch.stack(torch.meshgrid(t,T,S,rh,rf, indexing='ij'), dim=0)

In [7]:
# step 1 : monolithic samples, smooth surface


In [ ]:
T,S,rho = data_resice.get_scaled_cores(fn='RESICE/RESICE_PartA.csv', bins=125, device='cuda', min_samples=4, max_depth=2.5)

import brack2bing, data_resice
if (os.path.exists("brack2bing_solver_cache/init.pickle")):
    icesim = brack2bing.LatticeSolver.load()
else:
    icesim = brack2bing.LatticeSolver(device='cuda')
    icesim.save()
lattice = icesim.gen_lattice(T,S,rho, hshape=(1,25))
er = icesim.get_dielectric(lattice, T, 1.2)

samp = lattice[31,0,1,1:-1,1:-1].cpu()
samp = (samp > 0).to(torch.float32).mean(dim=0)
#samp = torch.fft.fft(samp); samp[30:]=0; samp = torch.fft.ifft(samp).real
plt.plot(samp); plt.xticks(np.linspace(0,125,30), np.linspace(0,250,30).round(), rotation=90); plt.xlabel('depth(cm)'); plt.title('brine frac'); plt.show()

import crefl2slc; importlib.reload(crefl2slc)
if (os.path.exists('SimpleReflector_Sentinel1_latest.pickle')):
    sref = crefl2slc.SimpleReflector.load('SimpleReflector_Sentinel1_latest.pickle')

samp = sref.refl[2,0,0,:].cpu().numpy()

pola = ['HH','HV','VH','VV']
for i in range(4):
    plt.scatter(samp[i].real, samp[i].imag)
    #plt.legend(pola[i])
    plt.annotate(
        pola[i],                      # The text label
        (samp[i].real, samp[i].imag),               # The data point coordinate (x, y)
        textcoords="offset points", # How to position the text
        xytext=(0, 10),             # Distance from text to point (x=0, y=10 points offset)
        ha='center'                 # Horizontal alignment (center, left, or right)
    )
plt.xlim(-0.2,0.2)
plt.ylim(-0.2,0.2)
plt.grid(True)
circu = np.array([np.sin(np.linspace(0,2*np.pi,50)), np.cos(np.linspace(0,2*np.pi,50))])*0.2
plt.plot(circu[1],circu[0])
plt.show()

print(np.rad2deg(np.angle(samp)), np.abs(samp))